# T2b — Duplicados y calidad de los datos
**Autores:** Noel (detección de duplicados) y Daniel (análisis, decisión y revisión de rangos)
**Proyecto:** P1 — Análisis exploratorio del Titanic
**Asignatura:** Python para IA — 2º DAM

## Objetivo

Comprobar si hay filas duplicadas en el dataset, decidir si se eliminan (justificando la decisión con datos) y revisar valores fuera de rango que puedan indicar errores de calidad. Trabajamos con los datos **originales**, sin tratar los nulos, para ver los duplicados tal y como vienen.

In [1]:
import pandas as pd
import numpy as np

# Carga del dataset desde la carpeta Data del repositorio.
# Si no se encuentra (por ejemplo al abrir el notebook en Colab), se descarga
# de la misma fuente original, así que los datos son idénticos.
try:
    df = pd.read_csv("../Data/titanic.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")

df.shape

(891, 15)

## 1. Detección de duplicados

In [2]:
# Comprobamos cuántas filas duplicadas hay
# (filas que son copia exacta de otra fila anterior, en todas las columnas)
print("Número de filas duplicadas:", df.duplicated().sum())

# Mostramos las filas que forman parte de un duplicado, incluida la primera aparición,
# para poder compararlas entre sí
mask_dup = df.duplicated(keep=False)
print("Filas implicadas en algún duplicado:", mask_dup.sum())
df[mask_dup].sort_values(["pclass", "sex", "fare"]).head(10)

Número de filas duplicadas: 107
Filas implicadas en algún duplicado: 160


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
369,1,1,female,24.0,0,0,69.3000,C,First,woman,False,B,Cherbourg,yes,True
641,1,1,female,24.0,0,0,69.3000,C,First,woman,False,B,Cherbourg,yes,True
64,0,1,male,NaN,0,0,27.7208,C,First,man,True,NaN,Cherbourg,no,True
295,0,1,male,NaN,0,0,27.7208,C,First,man,True,NaN,Cherbourg,no,True
458,1,2,female,50.0,0,0,10.5000,S,Second,woman,False,NaN,Southampton,yes,True
526,1,2,female,50.0,0,0,10.5000,S,Second,woman,False,NaN,Southampton,yes,True
443,1,2,female,28.0,0,0,13.0000,S,Second,woman,False,NaN,Southampton,yes,True
635,1,2,female,28.0,0,0,13.0000,S,Second,woman,False,NaN,Southampton,yes,True
53,1,2,female,29.0,1,0,26.0000,S,Second,woman,False,NaN,Southampton,yes,False
133,1,2,female,29.0,1,0,26.0000,S,Second,woman,False,NaN,Southampton,yes,False


## 2. ¿Son duplicados reales?

Este dataset (versión de seaborn) no incluye columnas que identifiquen a cada persona, como el nombre o el número de billete, que sí están en el CSV original de Kaggle. Por eso hay que comprobar si las filas "duplicadas" son errores de carga o pasajeros distintos que simplemente comparten los mismos valores.

In [3]:
# Tamaño de cada grupo de filas idénticas (dropna=False para no descartar
# los grupos que tienen NaN en age o deck)
grupos = (
    df[mask_dup]
    .groupby(list(df.columns), dropna=False)
    .size()
    .sort_values(ascending=False)
)
print("Grupos con exactamente 2 filas idénticas:", (grupos == 2).sum())
print("Grupos con 3 o más filas idénticas:", (grupos >= 3).sum())
print("Tamaño del grupo más grande:", grupos.max())

# ¿Cuántas de las filas duplicadas no tienen edad registrada?
print("Filas duplicadas sin edad (age = NaN):", df.loc[mask_dup, "age"].isna().sum(), "de", mask_dup.sum())

Grupos con exactamente 2 filas idénticas: 35
Grupos con 3 o más filas idénticas: 18
Tamaño del grupo más grande: 13
Filas duplicadas sin edad (age = NaN): 93 de 160


In [4]:
# El grupo más grande: ¿qué perfil de pasajero es?
grupos.head(3)

survived  pclass  sex   age  sibsp  parch  fare    embarked  class  who  adult_male  deck  embark_town  alive  alone
0         3       male  NaN  0      0      7.8958  S         Third  man  True        NaN   Southampton  no     True     13
                                           8.0500  S         Third  man  True        NaN   Southampton  no     True     12
                                           7.7500  Q         Third  man  True        NaN   Queenstown   no     True      8
dtype: int64

## 3. Impacto de eliminar los duplicados

In [5]:
# Comparamos el dataset completo con el que quedaría sin duplicados
tasa_con = df["survived"].mean() * 100
df_sin_dup = df.drop_duplicates()
tasa_sin = df_sin_dup["survived"].mean() * 100

print(f"Filas con todos los datos: {len(df)}")
print(f"Filas si elimináramos los duplicados: {len(df_sin_dup)}")
print(f"Tasa de supervivencia con todas las filas: {tasa_con:.2f}%")
print(f"Tasa de supervivencia sin duplicados: {tasa_sin:.2f}%")

Filas con todos los datos: 891
Filas si elimináramos los duplicados: 784
Tasa de supervivencia con todas las filas: 38.38%
Tasa de supervivencia sin duplicados: 41.20%


## 4. Valores fuera de rango

In [6]:
# Buscamos valores imposibles en las variables numéricas y categóricas
print("Edad mínima / máxima:", df["age"].min(), "/", df["age"].max())
print("Tarifa mínima / máxima:", df["fare"].min(), "/", df["fare"].max())
print("Valores únicos de pclass:", sorted(df["pclass"].unique()))
print("Valores únicos de sex:", df["sex"].unique())

# Pasajeros con tarifa 0: ¿son un error?
tarifa_cero = df[df["fare"] == 0]
print("\nPasajeros con tarifa = 0:", len(tarifa_cero))
tarifa_cero["pclass"].value_counts().sort_index()

Edad mínima / máxima: 0.42 / 80.0
Tarifa mínima / máxima: 0.0 / 512.3292
Valores únicos de pclass: [np.int64(1), np.int64(2), np.int64(3)]
Valores únicos de sex: <StringArray>
['male', 'female']
Length: 2, dtype: str

Pasajeros con tarifa = 0: 15


pclass
1    5
2    6
3    4
Name: count, dtype: int64

## 5. Decisión y justificación

**No se elimina ninguna fila.**

- Sin nombre ni número de billete, `duplicated()` solo compara 15 columnas, casi todas con pocos valores posibles (sexo, clase, puerto, etc.). Es esperable que pasajeros distintos coincidan en todas ellas.
- Los grupos de filas idénticas no son solo parejas: el más grande tiene **13 filas** (hombres de 3ª clase, sin edad registrada, viajando solos y con la misma tarifa barata de Southampton). Es un perfil muy habitual en el Titanic, no un error de carga que duplique a una misma persona.
- **93 de las 160 filas** implicadas en duplicados no tienen edad. La falta de edad elimina una de las pocas columnas que diferenciaría a esos pasajeros, lo que explica buena parte de las coincidencias.
- Eliminarlas quitaría el 12% del dataset (de 891 a 784 filas) y cambiaría la tasa de supervivencia global de 38,38% a 41,20%. Como las filas repetidas son sobre todo hombres de 3ª clase que no sobrevivieron, borrarlas sesgaría el análisis en vez de corregirlo.
- No se han encontrado valores imposibles: edades entre 0,42 y 80 años, tarifas entre 0 y 512,33, y `pclass`/`sex` solo con los valores esperados. Hay 15 pasajeros con tarifa 0, repartidos entre las tres clases; lo más probable es que viajaran con billete gratuito o de cortesía, así que no se tratan como error.

No podemos asegurar al 100% que ninguna fila sea un duplicado real, porque faltan los identificadores. Pero con los datos disponibles, eliminarlas causaría más distorsión de la que evitaría.

## Hallazgo

Hay 107 filas duplicadas (160 filas implicadas en total), pero se decidió **no eliminarlas**. El dataset no tiene columnas identificadoras y los grupos de filas idénticas, de hasta 13 pasajeros, responden a perfiles comunes (sobre todo hombres de 3ª clase viajando solos, 93 de ellos sin edad registrada). Eliminarlas reduciría el dataset un 12% y subiría la tasa de supervivencia global de 38,38% a 41,20%, sesgando el análisis. No se encontraron valores fuera de rango.